# Can the limit order book predict short-term price moves?

**Question.** After a meaningful price move, can order book data predict the next move (down, time-out, up), and is it worth trading after the spread?

Slides: [`reports/order_book_presentation.pdf`](../../reports/order_book_presentation.pdf). Write-up: [`reports/order_book_write_up.pdf`](../../reports/order_book_write_up.pdf).

- **Data:** FI-2010 limit order book, 5 stocks, 10 days (Ntakaris et al., 2018).
- **Target:** triple-barrier label per CUSUM event (3 classes).
- **Features:** 40 book columns, 14 engineered (spread, depth imbalance, microprice, OFI), 104 handcrafted, stock one-hot.
- **Models:** logistic regression, MLP small, MLP large, LSTM, CNN+LSTM.
- **Validation:** purged walk-forward, 4 folds per stock. Test files used once at the end.

| Setup | Labels | Loss |
|---|---|---|
| `main` | Volatility-scaled barriers, 20 ticks | Cross-entropy |
| `cost_labels` | Barriers ≥ 2 spreads, longer horizon | Cross-entropy |
| `cost_loss` | As `cost_labels` | Cross-entropy − expected net return |

| Section | Content |
|---|---|
| 1 | Setup and settings |
| 2 | Load, describe, segment, folds |
| 3 | Features, events, labels, weights |
| 4 | Model inputs and models |
| 5 | Pipeline: tune, compare, select, calibrate |
| 6 | Run the three setups |
| 7 | Test |
| 8 | Business scenarios |
| 9 | Diagnostics |
| 10 | Key results and export |

Every table is displayed and saved as a numbered CSV for Excel.

## 1. Setup and settings

- `RETRAIN_MODELS`: ignore cached tuning results and weights.
- `EVALUATE_TEST`: score the test files (logged in `test_evaluations.csv`).
- `DOWNLOAD_EXPORTS`: download all CSVs as one zip at the end.
- Cached files are named by a fingerprint of settings + code, so any change retrains instead of reusing stale results.
- Training: AdamW, ReduceLROnPlateau (×0.5, patience 2), gradient clip 1.0, early stopping (patience 8, max 60 epochs).

In [ ]:
!pip install -q optuna wandb
!wget -nc https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip
!unzip -n data.zip

In [ ]:
import copy
import hashlib
import json
import platform
import shutil
import types
import zlib
from dataclasses import asdict, dataclass, field, replace
from datetime import datetime
from enum import IntEnum
from functools import cached_property
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import sklearn
import statsmodels
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller

try:
    from google.colab import drive, files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# Run switches
RETRAIN_MODELS = False
EVALUATE_TEST = False  # turn on only after choosing a model from the walk-forward results
DOWNLOAD_EXPORTS = True  # on Colab, download every CSV as one zip at the end

# Everything saved goes in this folder, inside My Drive on Colab
PROJECT_SUBFOLDER = 'order-book-prediction/artifacts'
WANDB_PROJECT = 'fi2010-order-book'


@dataclass(frozen=True)
class PipelineConfig:
    """All data settings."""
    train_file: str = 'Train_Dst_NoAuction_DecPre_CF_7.txt'
    test_files: dict = field(default_factory=lambda: {
        'Test_7': 'Test_Dst_NoAuction_DecPre_CF_7.txt',
        'Test_8': 'Test_Dst_NoAuction_DecPre_CF_8.txt',
        'Test_9': 'Test_Dst_NoAuction_DecPre_CF_9.txt',
    })
    train_stock_boundaries: tuple = (23336, 68210, 106759, 161434)
    jump_threshold: float = 0.0075  # test files never jump more than about 0.5%
    d_candidates: tuple = tuple(round(0.10 + 0.05 * i, 2) for i in range(19))
    min_stationary_share: float = 0.9
    adf_significance: float = 0.05
    weight_cutoff: float = 1e-4
    volatility_window: int = 100
    cusum_threshold_multiplier: float = 2.5  # chosen by hand; section 3 shows the alternatives
    cusum_report_candidates: tuple = (0.5, 1.0, 2.5)
    min_threshold: float = 1e-5  # floor for the CUSUM threshold and the barrier width
    barrier_candidates: tuple = (1.0, 2.5)
    min_barrier_spreads: float = 0.0  # barriers at least this many spreads wide (cost-aware setups)
    target_timeout_range: tuple = (0.20, 0.35)
    holding_period: int = 20
    fold_edges: tuple = (0.5, 0.6, 0.7, 0.8, 1.0)  # share of each stock's ticks
    embargo_fraction: float = 0.01
    lookback: int = 100


@dataclass(frozen=True)
class ModelConfig:
    """Model and training settings. Tuning and final training share all of them."""
    seeds: tuple = (42, 43, 44)
    batch_size: int = 256
    max_epochs: int = 60
    patience: int = 8  # epochs without improvement before stopping
    lr_decay_factor: float = 0.5
    lr_decay_patience: int = 2  # epochs without improvement before halving the learning rate
    min_learning_rate: float = 1e-6
    gradient_clip: float = 1.0
    tuning_trials: int = 20
    tuning_startup_trials: int = 5
    tuning_folds: tuple = (1, 2)  # folds 2 and 3, counting from zero
    learning_rate: float = 0.001
    weight_decay: float = 1e-5
    mlp_small_dims: tuple = (128, 64)
    mlp_large_dims: tuple = (256, 128, 64)
    dropout: float = 0.2
    lstm_hidden_dim: int = 32
    lstm_layers: int = 2
    cnn_out_channels: int = 16
    cnn_kernel_size: int = 3
    objective: str = 'cross_entropy'  # or 'cost_sensitive'
    cost_loss_weight: float = 0.1


COST_AWARE_SPREADS = 2.0
COST_AWARE_HOLDING_CANDIDATES = (50, 100, 200, 400)

N_STOCKS = 5
SCALING_MODES = ('global', 'per_stock')
LEVELS = range(1, 11)

# File order for each of the 10 levels: ask price, ask size, bid price, bid size.
BOOK_COLUMNS = [f'{name}_{level}' for level in LEVELS for name in ('ask_price', 'ask_size', 'bid_price', 'bid_size')]
PRICE_COLUMNS = [column for column in BOOK_COLUMNS if '_price_' in column]
HANDCRAFTED_COLUMNS = [f'handcrafted_{index}' for index in range(104)]
ORIGINAL_LABEL_COLUMNS = ['label_k10', 'label_k20', 'label_k30', 'label_k50', 'label_k100']
ENGINEERED_COLUMNS = ['frac_mid_price', 'spread', 'depth_imbalance', 'microprice_distance', 'ofi']
DEEP_OFI_COLUMNS = [f'ofi_{level}' for level in LEVELS if level > 1]
STOCK_COLUMNS = [f'stock_{stock_id}' for stock_id in range(N_STOCKS)]  # one-hot, not scaled
BASE_COLUMNS = BOOK_COLUMNS + ENGINEERED_COLUMNS

# Chosen on walk-forward validation loss in an earlier run (table in section 4)
selected_scaling = 'per_stock'
selected_feature_set = 'with_handcrafted'
selected_columns = BASE_COLUMNS + DEEP_OFI_COLUMNS + HANDCRAFTED_COLUMNS

CLASS_NAMES = ['down', 'time-out', 'up']
NUM_CLASSES = len(CLASS_NAMES)
BASELINES = ('majority_class', 'always_up', 'always_down')
SCORE_COLUMNS = [
    'accuracy', 'weighted_accuracy', 'macro_f1', 'trade_share', 'direction_precision', 'gross_bps', 'net_bps',
]

config = PipelineConfig()
model_config = ModelConfig()

In [ ]:
@dataclass(frozen=True)
class RunContext:
    device: torch.device
    project_dir: Path
    data_dir: Path
    weights_dir: Path
    tuning_dir: Path
    run_dir: Path
    wandb_enabled: bool


def resolve_project_dir(subfolder):
    """My Drive on Colab, so a shared notebook uses each person's own Drive."""
    if IN_COLAB:
        drive.mount('/content/drive')
        base = Path('/content/drive/MyDrive')
    else:
        base = Path.cwd()
    project_dir = base / subfolder
    project_dir.mkdir(parents=True, exist_ok=True)
    return project_dir


def login_to_wandb():
    """Use the Colab secret named 'wandb' if there is one. Everything still runs without it."""
    if not IN_COLAB:
        return False
    try:
        wandb.login(key=userdata.get('wandb'))
        return True
    except Exception as error:
        print(f'Running without Weights & Biases: {error}')
        return False


def create_context(subfolder):
    project_dir = resolve_project_dir(subfolder)
    folders = {name: project_dir / name for name in ('data', 'weights', 'tuning', 'runs')}
    for folder in folders.values():
        folder.mkdir(exist_ok=True)
    run_dir = folders['runs'] / datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir.mkdir()
    return RunContext(
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        project_dir=project_dir,
        data_dir=folders['data'],
        weights_dir=folders['weights'],
        tuning_dir=folders['tuning'],
        run_dir=run_dir,
        wandb_enabled=login_to_wandb(),
    )


def code_object_text(code):
    parts = [code.co_code.hex(), ','.join(code.co_names)]
    for constant in code.co_consts:
        if isinstance(constant, types.CodeType):
            parts.append(code_object_text(constant))
        elif isinstance(constant, frozenset):
            parts.append(repr(sorted(repr(item) for item in constant)))
        else:
            parts.append(repr(constant))
    return '|'.join(parts)


def code_text(item):
    """Text that changes when a function's or class's code changes."""
    if isinstance(item, type):
        return item.__name__ + ''.join(code_text(member) for _, member in sorted(vars(item).items()))
    if isinstance(item, (staticmethod, classmethod)):
        return code_text(item.__func__)
    if isinstance(item, property):
        return code_text(item.fget)
    if isinstance(item, cached_property):
        return code_text(item.func)
    if isinstance(item, types.FunctionType):
        return code_object_text(item.__code__)
    return ''


def fingerprint(settings, code_items=()):
    """Short id for a set of settings and code."""
    text = json.dumps(settings, sort_keys=True, default=str) + ''.join(code_text(item) for item in code_items)
    return hashlib.sha256(text.encode()).hexdigest()[:12]


def cached_json(path, compute):
    """Reuse a saved result unless RETRAIN_MODELS is on."""
    if path.exists() and not RETRAIN_MODELS:
        return json.loads(path.read_text())
    value = compute()
    path.write_text(json.dumps(value, indent=2, default=float))
    return value


class Exporter:
    """Shows tables and saves each one as a numbered CSV, with an index of chart ideas."""

    def __init__(self, folder):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.rows = []

    def save(self, name, table, chart_idea, title=None, show=True):
        path = self.folder / f'{len(self.rows) + 1:02d}_{name}.csv'
        table.to_csv(path, index=False)
        self.rows.append({'file': path.name, 'rows': len(table), 'chart_idea': chart_idea})
        if show:
            print(title or name.replace('_', ' ').capitalize())
            display(table.head(40).round(4))

    def finish(self):
        """Write the index and zip the folder. Returns the zip's path."""
        pd.DataFrame(self.rows).to_csv(self.folder / '00_index.csv', index=False)
        zip_path = shutil.make_archive(str(self.folder), 'zip', self.folder)
        print(f'Saved {len(self.rows)} CSV files to {self.folder}, zipped as {Path(zip_path).name}')
        return zip_path


context = create_context(PROJECT_SUBFOLDER)
exporter = Exporter(context.run_dir / 'excel_exports')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f'Device: {context.device}')
print(f'Run folder: {context.run_dir}')
print(f'Weights & Biases: {"on" if context.wandb_enabled else "off"}')

## 2. Load, describe, segment, folds

- **Files:** 1 train file (days 1-7), 3 test files (days 8-10), normalized, no auction. Stored transposed (149 rows × ticks); cached as pickle after the first read.
- **Row:** one book update = 40 book columns (10 levels × ask/bid × price/size) + 104 handcrafted + 5 original labels (never used as features).
- **Segments:** each file stacks 5 stocks. Cut at known train stock boundaries and at mid-price jumps > 0.75%. All rolling calculations stay inside a segment. Each test file must give exactly 5 segments.
- **Folds:** expanding walk-forward per stock (train 50/60/70/80%, validate the next 10/10/10/20%). Purge: train labels end before the cut. Embargo: max(holding period, 1% of the stock's ticks) (Lopez de Prado, 2018).
- **Settings period:** first 50% of each stock. `d`, barrier width and holding period are chosen only here.

In [ ]:
class LobFile:
    """One FI-2010 file with named columns."""

    def __init__(self, name, book, handcrafted, original_labels):
        self.name = name
        self.book = book
        self.handcrafted = handcrafted
        self.original_labels = original_labels
        self.segments = []  # filled when segmenting

    @classmethod
    def from_raw(cls, raw, name):
        book = raw.iloc[:, :40].set_axis(BOOK_COLUMNS, axis=1)
        handcrafted = raw.iloc[:, 40:144].set_axis(HANDCRAFTED_COLUMNS, axis=1)
        original_labels = raw.iloc[:, -5:].set_axis(ORIGINAL_LABEL_COLUMNS, axis=1)
        return cls(name, book, handcrafted, original_labels)

    @cached_property
    def mid_price(self):
        return compute_mid_price(self.book)

    @cached_property
    def relative_spread(self):
        """Spread / mid-price, the cost of crossing the spread once."""
        return (self.book['ask_price_1'].to_numpy() - self.book['bid_price_1'].to_numpy()) / self.mid_price


def compute_mid_price(book):
    return (book['ask_price_1'].to_numpy() + book['bid_price_1'].to_numpy()) / 2.0


def read_raw_file(path, cache_dir):
    cache_path = cache_dir / f'{Path(path).stem}.pkl'
    if cache_path.exists():
        return pd.read_pickle(cache_path)
    raw = pd.read_csv(path, sep=r'\s+', header=None).T.astype(float)
    raw.to_pickle(cache_path)
    return raw


train_file = LobFile.from_raw(read_raw_file(config.train_file, context.data_dir), 'train_file')
test_files = {
    name: LobFile.from_raw(read_raw_file(path, context.data_dir), name)
    for name, path in config.test_files.items()
}
all_files = {'train_file': train_file, **test_files}

exporter.save(
    'file_sizes',
    pd.DataFrame([
        {'file': name, 'rows': len(file.book), 'book_columns': file.book.shape[1],
         'handcrafted_columns': file.handcrafted.shape[1], 'missing_values': int(file.book.isna().sum().sum())}
        for name, file in all_files.items()
    ]),
    'Table for the data section',
)


def describe_columns(table):
    return table.describe().T.reset_index().rename(columns={'index': 'column'})


raw_view = train_file.book.assign(mid_price=train_file.mid_price, relative_spread=train_file.relative_spread)
exporter.save('raw_descriptive_stats', describe_columns(raw_view), 'Table of mean and standard deviation for the data section')

In [ ]:
@dataclass(frozen=True)
class Segment:
    """A stretch of one stock with no jump above the threshold. `end` is exclusive."""
    stock_id: int
    start: int
    end: int
    median_mid_price: float = 1.0
    median_spread: float = 0.0
    largest_jump: float = 0.0


def detect_segment_ranges(mid_price, jump_threshold, stock_boundaries=()):
    jumps = np.abs(np.diff(mid_price)) / mid_price[:-1]
    jump_ticks = (np.flatnonzero(jumps > jump_threshold) + 1).tolist()
    edges = sorted({0, len(mid_price), *jump_ticks, *stock_boundaries})
    return list(zip(edges[:-1], edges[1:]))


def assign_stock_ids(ranges, stock_boundaries, dataset_name):
    """Use known boundaries if given. Otherwise require exactly one segment per stock."""
    if len(stock_boundaries) > 0:
        return [int(np.searchsorted(stock_boundaries, start, side='right')) for start, _ in ranges]
    if len(ranges) != N_STOCKS:
        raise ValueError(f'{dataset_name}: expected {N_STOCKS} segments, found {len(ranges)}')
    return list(range(N_STOCKS))


def build_segments(file, jump_threshold, stock_boundaries=()):
    ranges = detect_segment_ranges(file.mid_price, jump_threshold, stock_boundaries)
    stock_ids = assign_stock_ids(ranges, stock_boundaries, file.name)
    spread = file.relative_spread * file.mid_price
    segments = []
    for (start, end), stock_id in zip(ranges, stock_ids):
        segment_mid = file.mid_price[start:end]
        jumps = np.abs(np.diff(segment_mid)) / segment_mid[:-1]
        segments.append(Segment(
            stock_id, start, end, float(np.median(segment_mid)),
            float(np.median(spread[start:end])), float(jumps.max()) if len(jumps) else 0.0,
        ))
    return segments


def check_test_segments(train_segments, test_files):
    """Each test segment should be priced near the same stock's last training segment."""
    last_train = {segment.stock_id: segment for segment in train_segments}
    for name, file in test_files.items():
        assert len(file.segments) == N_STOCKS, name
        for segment in file.segments:
            reference = last_train[segment.stock_id].median_mid_price
            gap = abs(segment.median_mid_price - reference) / reference
            assert gap <= 0.15, (name, segment.stock_id, gap)
    print('Each test file has exactly 5 segments, priced near the matching training stock')


def stock_statistics(file):
    """Mean and standard deviation of key values for each stock."""
    stock_ids = np.empty(len(file.book), dtype=int)
    for segment in file.segments:
        stock_ids[segment.start:segment.end] = segment.stock_id
    view = pd.DataFrame({
        'mid_price': file.mid_price,
        'spread_bps': file.relative_spread * 1e4,
        'ask_size_1': file.book['ask_size_1'].to_numpy(),
        'bid_size_1': file.book['bid_size_1'].to_numpy(),
    })
    table = view.groupby(stock_ids).agg(['mean', 'std'])
    table.columns = ['_'.join(column) for column in table.columns]
    return table.reset_index().rename(columns={'index': 'stock_id'})


train_file.segments = build_segments(train_file, config.jump_threshold, config.train_stock_boundaries)
for file in test_files.values():
    file.segments = build_segments(file, config.jump_threshold)
check_test_segments(train_file.segments, test_files)

exporter.save(
    'segments',
    pd.DataFrame([
        {'file': name, **asdict(segment), 'length': segment.end - segment.start}
        for name, file in all_files.items() for segment in file.segments
    ]),
    'Table, or bars of largest_jump by segment',
    'Segments. Stocks 1 and 3 trade at similar prices, so look over those rows.',
)
exporter.save('stock_statistics', stock_statistics(train_file), 'Clustered bars of spread_bps_mean by stock')

In [ ]:
@dataclass(frozen=True)
class StockFold:
    first_tick: int
    train_end: int
    validation_start: int
    validation_end: int


def compute_folds(segments, config):
    """A list of folds, each mapping stock id to its time ranges.

    Assumes each stock's segments are chronological in file order. The files have no timestamps.
    """
    edges = config.fold_edges
    folds = [{} for _ in range(len(edges) - 1)]
    for stock_id in sorted({segment.stock_id for segment in segments}):
        stock_segments = [segment for segment in segments if segment.stock_id == stock_id]
        first_tick = min(segment.start for segment in stock_segments)
        tick_count = max(segment.end for segment in stock_segments) - first_tick
        gap = max(config.holding_period, int(tick_count * config.embargo_fraction))
        for index, fold in enumerate(folds):
            train_end = first_tick + int(edges[index] * tick_count)
            validation_end = first_tick + int(edges[index + 1] * tick_count)
            fold[stock_id] = StockFold(first_tick, train_end, train_end + gap, validation_end)
    return folds


def in_train_period(ticks, fold):
    mask = np.zeros(len(ticks), dtype=bool)
    for stock_fold in fold.values():
        mask |= (ticks >= stock_fold.first_tick) & (ticks < stock_fold.train_end)
    return mask


def clip_segments(segments, fold):
    """Cut segments at each stock's train_end, keeping only the part before it."""
    clipped = []
    for segment in segments:
        end = min(segment.end, fold[segment.stock_id].train_end)
        if end > segment.start:
            clipped.append(replace(segment, end=end))
    return clipped


folds = compute_folds(train_file.segments, config)
settings_fold = folds[0]
settings_segments = clip_segments(train_file.segments, settings_fold)

exporter.save(
    'walk_forward_folds',
    pd.DataFrame([
        {'fold': index + 1, 'stock_id': stock_id, **asdict(stock_fold)}
        for index, fold in enumerate(folds) for stock_id, stock_fold in fold.items()
    ]),
    'Table for the methods section',
)

## 3. Features, events, labels, weights

All features use only current and past ticks inside a segment.

- **Fractional differentiation:** smallest `d` (step 0.05) with ≥ 90% of settings-period segments ADF-stationary at 5%. Keeps more memory than returns (Lopez de Prado, 2018). Warm-up rows are dropped.
- **Features:** book prices as distance from mid; spread; depth imbalance; microprice − mid; OFI at levels 1-10 (Cont, Kukanov and Stoikov, 2014); 104 handcrafted (not recomputed).
- **Events:** symmetric CUSUM filter, threshold 2.5 × rolling volatility (set by hand; alternatives shown). Each event needs a full 100-tick lookback and a full holding period inside its segment.
- **Labels:** triple barrier, width = multiplier × volatility (floored at `min_barrier_spreads` × spread). Up/down = first barrier hit; time-out = neither within the holding period. Multiplier chosen for 20-35% time-outs.
- **Weights:** average uniqueness over each label's life, so overlapping events count less.
- **Returns and costs:** log mid return t0→t1, and relative spread at t0 as the cost of one crossing.

In [ ]:
class FractionalDifferentiator:
    """Fractional differentiation with a fixed window of weights."""

    def __init__(self, d, weight_cutoff):
        self.d = d
        self.weights = self._compute_weights(d, weight_cutoff)

    @staticmethod
    def _compute_weights(d, weight_cutoff):
        weights = [1.0]
        k = 1
        while True:
            weight = -weights[-1] / k * (d - k + 1)
            if abs(weight) < weight_cutoff:
                return np.array(weights)
            weights.append(weight)
            k += 1

    @property
    def warmup_length(self):
        return len(self.weights)

    def transform(self, values):
        """Uses only current and past values. The caller drops the warm-up rows."""
        return np.convolve(values, self.weights)[:len(values)]


def adf_p_value(series):
    # result_object=False needs the newer statsmodels that Colab has, so run this in Colab.
    return adfuller(series, autolag='AIC', result_object=False)[1]


def search_d(mid_price, segments, config):
    """Try candidates from smallest to largest and stop at the first that passes."""
    rows = []
    for d in config.d_candidates:
        frac_diff = FractionalDifferentiator(d, config.weight_cutoff)
        series = [frac_diff.transform(mid_price[s.start:s.end])[frac_diff.warmup_length:] for s in segments]
        tested = [values for values in series if len(values) > 30]
        passed = sum(adf_p_value(values) <= config.adf_significance for values in tested)
        rows.append({
            'd': d, 'stationary_segments': passed, 'tested_segments': len(tested),
            'warmup_length': frac_diff.warmup_length,
        })
        if passed / len(tested) >= config.min_stationary_share:
            return rows
    raise ValueError(f'No d candidate reached a stationary share of {config.min_stationary_share}')


def compute_order_flow_imbalance(bid_price, bid_size, ask_price, ask_size):
    """Order flow imbalance at one level. The first tick is 0 because it has no previous tick."""
    previous_bid_price = np.r_[bid_price[:1], bid_price[:-1]]
    previous_bid_size = np.r_[bid_size[:1], bid_size[:-1]]
    previous_ask_price = np.r_[ask_price[:1], ask_price[:-1]]
    previous_ask_size = np.r_[ask_size[:1], ask_size[:-1]]
    bid_flow = np.where(
        bid_price > previous_bid_price, bid_size,
        np.where(bid_price == previous_bid_price, bid_size - previous_bid_size, -previous_bid_size),
    )
    ask_flow = np.where(
        ask_price < previous_ask_price, ask_size,
        np.where(ask_price == previous_ask_price, ask_size - previous_ask_size, -previous_ask_size),
    )
    return bid_flow - ask_flow


def level_ofi(segment_book, level):
    return compute_order_flow_imbalance(
        segment_book[f'bid_price_{level}'].to_numpy(), segment_book[f'bid_size_{level}'].to_numpy(),
        segment_book[f'ask_price_{level}'].to_numpy(), segment_book[f'ask_size_{level}'].to_numpy(),
    )


def build_segment_features(segment_book, frac_diff):
    """All features for one segment, including its warm-up rows."""
    ask_price = segment_book['ask_price_1'].to_numpy()
    ask_size = segment_book['ask_size_1'].to_numpy()
    bid_price = segment_book['bid_price_1'].to_numpy()
    bid_size = segment_book['bid_size_1'].to_numpy()
    mid_price = compute_mid_price(segment_book)
    total_size = bid_size + ask_size + 1e-8

    features = segment_book.copy()
    features[PRICE_COLUMNS] = segment_book[PRICE_COLUMNS].to_numpy() - mid_price[:, None]
    features['frac_mid_price'] = frac_diff.transform(mid_price)
    features['spread'] = ask_price - bid_price
    features['depth_imbalance'] = (bid_size - ask_size) / total_size
    features['microprice_distance'] = (bid_price * ask_size + ask_price * bid_size) / total_size - mid_price
    features['ofi'] = level_ofi(segment_book, 1)
    for level in LEVELS:
        if level > 1:
            features[f'ofi_{level}'] = level_ofi(segment_book, level)
    return features


def build_tick_features(file, frac_diff):
    """Unscaled features for all ticks, indexed by tick, without each segment's warm-up rows."""
    frames = [
        build_segment_features(file.book.iloc[segment.start:segment.end], frac_diff).iloc[frac_diff.warmup_length:]
        for segment in file.segments
    ]
    features = pd.concat(frames)
    handcrafted = file.handcrafted.loc[features.index].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([features, handcrafted], axis=1)


d_settings = {key: value for key, value in asdict(config).items() if key != 'test_files'}
d_rows = cached_json(
    context.tuning_dir / f'd_search_{fingerprint(d_settings, [search_d, build_segments, compute_folds])}.json',
    lambda: search_d(train_file.mid_price, settings_segments, config),
)
selected_d = d_rows[-1]['d']
frac_diff = FractionalDifferentiator(selected_d, config.weight_cutoff)
exporter.save('d_search', pd.DataFrame(d_rows), 'Line chart of stationary_segments by d', 'd search on the settings period')
print(f'Selected d: {selected_d}')

first_segment = train_file.mid_price[train_file.segments[0].start:train_file.segments[0].end]
exporter.save(
    'stationarity_example',
    pd.DataFrame([
        {'series': 'mid_price', 'adf_p_value': adf_p_value(first_segment)},
        {'series': 'log_returns', 'adf_p_value': adf_p_value(np.diff(np.log(first_segment)))},
        {'series': f'frac_diff_d={selected_d}',
         'adf_p_value': adf_p_value(frac_diff.transform(first_segment)[frac_diff.warmup_length:])},
    ]),
    'Table',
    'Stationarity test on the first training segment. Below 0.05 means stationary.',
)

In [ ]:
class BarrierLabel(IntEnum):
    DOWN = -1
    TIMEOUT = 0
    UP = 1


def compute_rolling_volatility(mid_price, segments, window):
    """Rolling std of log returns using past data inside each segment. The first tick stays NaN."""
    volatility = np.full(len(mid_price), np.nan)
    for segment in segments:
        segment_mid = mid_price[segment.start:segment.end]
        returns = np.zeros(len(segment_mid))
        returns[1:] = np.log(segment_mid[1:] / segment_mid[:-1])
        volatility[segment.start:segment.end] = pd.Series(returns).rolling(window, min_periods=2).std().to_numpy()
    return volatility


def detect_cusum_events(mid_price, volatility, segments, threshold_multiplier, min_threshold):
    """Symmetric CUSUM filter: the threshold scales with volatility and is only the trigger."""
    event_ticks = []
    for segment in segments:
        positive_sum = 0.0
        negative_sum = 0.0
        for t in range(segment.start + 1, segment.end):
            log_return = np.log(mid_price[t] / mid_price[t - 1])
            threshold = max(threshold_multiplier * volatility[t], min_threshold)
            positive_sum = max(0.0, positive_sum + log_return)
            negative_sum = min(0.0, negative_sum + log_return)
            if positive_sum > threshold:
                positive_sum = 0.0
                event_ticks.append(t)
            elif negative_sum < -threshold:
                negative_sum = 0.0
                event_ticks.append(t)
    return np.array(event_ticks, dtype=int)


def filter_eligible_events(event_ticks, segments, warmup_length, lookback, holding_period):
    """Keep events with a full lookback window after warm-up, and a full holding period before the segment ends."""
    keep = np.zeros(len(event_ticks), dtype=bool)
    for segment in segments:
        first_eligible_tick = segment.start + warmup_length + lookback - 1
        keep |= (event_ticks >= first_eligible_tick) & (event_ticks < segment.end - holding_period)
    return event_ticks[keep]


def barrier_widths(volatility, relative_spread, multiplier, config):
    """Barrier width in log-return terms, with floors for flat prices and for trading costs."""
    return np.maximum.reduce([
        multiplier * volatility,
        config.min_barrier_spreads * relative_spread,
        np.full(len(volatility), config.min_threshold),
    ])


def find_first_barrier_touch(mid_price, t0, segment_end, width, holding_period):
    """Return (t1, label). t1 is the first barrier touch or the vertical barrier."""
    upper_barrier = mid_price[t0] * np.exp(width)
    lower_barrier = mid_price[t0] * np.exp(-width)
    vertical_barrier = min(t0 + holding_period, segment_end - 1)
    for t in range(t0 + 1, vertical_barrier + 1):
        if mid_price[t] >= upper_barrier:
            return t, BarrierLabel.UP
        if mid_price[t] <= lower_barrier:
            return t, BarrierLabel.DOWN
    return vertical_barrier, BarrierLabel.TIMEOUT


def label_events(event_ticks, mid_price, widths, segments, holding_period):
    """Returns a DataFrame indexed by event tick t0, with t1, label and stock_id."""
    rows = []
    for segment in segments:
        segment_ticks = event_ticks[(event_ticks >= segment.start) & (event_ticks < segment.end)]
        for t0 in segment_ticks.tolist():
            t1, label = find_first_barrier_touch(mid_price, t0, segment.end, widths[t0], holding_period)
            rows.append((t0, t1, int(label), segment.stock_id))
    if len(rows) != len(event_ticks):
        raise ValueError('Every event must fall inside a segment')
    return pd.DataFrame(rows, columns=['t0', 't1', 'label', 'stock_id']).astype(int).set_index('t0')


def compute_class_shares(labels):
    return {
        'down_share': float(np.mean(labels == BarrierLabel.DOWN)),
        'timeout_share': float(np.mean(labels == BarrierLabel.TIMEOUT)),
        'up_share': float(np.mean(labels == BarrierLabel.UP)),
    }


def sample_events(file, volatility, frac_diff, config):
    ticks = detect_cusum_events(
        file.mid_price, volatility, file.segments, config.cusum_threshold_multiplier, config.min_threshold,
    )
    return filter_eligible_events(ticks, file.segments, frac_diff.warmup_length, config.lookback, config.holding_period)


def settings_period_events(event_ticks, holding_period, fold):
    """Events whose whole label window falls inside the settings period."""
    return event_ticks[in_train_period(event_ticks, fold) & in_train_period(event_ticks + holding_period, fold)]


def tune_barriers(file, event_ticks, volatility, config):
    """Choose the multiplier with time-outs closest to the middle of the target range."""
    low, high = config.target_timeout_range
    rows = []
    for multiplier in config.barrier_candidates:
        widths = barrier_widths(volatility, file.relative_spread, multiplier, config)
        events = label_events(event_ticks, file.mid_price, widths, file.segments, config.holding_period)
        shares = compute_class_shares(events['label'].to_numpy())
        rows.append({
            'barrier_multiplier': multiplier, **shares,
            'in_target_range': bool(low <= shares['timeout_share'] <= high),
        })
    best = min(rows, key=lambda row: abs(row['timeout_share'] - (low + high) / 2))
    return best['barrier_multiplier'], rows


def compare_cusum_thresholds(file, volatility, config, fold):
    moves = np.diff(file.mid_price)[in_train_period(np.arange(1, len(file.mid_price)), fold)]
    price_moves = np.count_nonzero(moves)
    rows = []
    for multiplier in config.cusum_report_candidates:
        ticks = detect_cusum_events(file.mid_price, volatility, file.segments, multiplier, config.min_threshold)
        count = int(in_train_period(ticks, fold).sum())
        rows.append({
            'cusum_multiplier': multiplier, 'events': count, 'events_per_price_move': count / price_moves,
            'selected': multiplier == config.cusum_threshold_multiplier,
        })
    return rows


train_volatility = compute_rolling_volatility(train_file.mid_price, train_file.segments, config.volatility_window)
exporter.save(
    'cusum_comparison', pd.DataFrame(compare_cusum_thresholds(train_file, train_volatility, config, settings_fold)),
    'Bars of events_per_price_move by threshold',
    'CUSUM thresholds on the settings period. Near 1.0 would mean almost every price move is an event.',
)
selected_barrier, barrier_rows = tune_barriers(
    train_file,
    settings_period_events(sample_events(train_file, train_volatility, frac_diff, config), config.holding_period, settings_fold),
    train_volatility, config,
)
exporter.save('barrier_comparison', pd.DataFrame(barrier_rows), 'Stacked bars of class shares by barrier', 'Barrier multipliers on the settings period')
print(f'Selected barrier multiplier: {selected_barrier}')

In [ ]:
def compute_uniqueness(t0, t1, n_ticks):
    """Average of 1 / concurrency over each event's life, from t0 to t1 inclusive."""
    changes = np.zeros(n_ticks + 1, dtype=int)
    np.add.at(changes, t0, 1)
    np.add.at(changes, t1 + 1, -1)
    concurrency = np.cumsum(changes)[:n_ticks]
    inverse = np.zeros(n_ticks)
    active = concurrency > 0
    inverse[active] = 1.0 / concurrency[active]
    running_total = np.r_[0.0, np.cumsum(inverse)]
    return (running_total[t1 + 1] - running_total[t0]) / (t1 - t0 + 1)


def stock_ids_for_ticks(ticks, segments):
    stock_ids = np.full(len(ticks), -1, dtype=int)
    for segment in segments:
        stock_ids[(ticks >= segment.start) & (ticks < segment.end)] = segment.stock_id
    return stock_ids


@dataclass(frozen=True, eq=False)
class PreparedFile:
    """Unscaled tick features and labeled events for one file."""
    name: str
    features: pd.DataFrame
    stock_ids: np.ndarray
    events: pd.DataFrame


def prepare_file(file, frac_diff, barrier_multiplier, config):
    features = build_tick_features(file, frac_diff)
    volatility = compute_rolling_volatility(file.mid_price, file.segments, config.volatility_window)
    event_ticks = sample_events(file, volatility, frac_diff, config)
    widths = barrier_widths(volatility, file.relative_spread, barrier_multiplier, config)
    events = label_events(event_ticks, file.mid_price, widths, file.segments, config.holding_period)
    t0 = events.index.to_numpy()
    t1 = events['t1'].to_numpy()
    events['weight'] = compute_uniqueness(t0, t1, len(file.book))
    events['gross_return'] = np.log(file.mid_price[t1] / file.mid_price[t0])
    events['cost'] = file.relative_spread[t0]
    return PreparedFile(file.name, features, stock_ids_for_ticks(features.index.to_numpy(), file.segments), events)


def event_summary(prepared_files):
    return pd.DataFrame([
        {
            'file': prepared.name,
            'events': len(prepared.events),
            **compute_class_shares(prepared.events['label'].to_numpy()),
            'average_uniqueness': prepared.events['weight'].mean(),
            'return_mean_bps': prepared.events['gross_return'].mean() * 1e4,
            'return_std_bps': prepared.events['gross_return'].std() * 1e4,
            'cost_mean_bps': prepared.events['cost'].mean() * 1e4,
            'cost_std_bps': prepared.events['cost'].std() * 1e4,
        }
        for prepared in prepared_files
    ])


prepared_train = prepare_file(train_file, frac_diff, selected_barrier, config)
prepared_tests = {name: prepare_file(file, frac_diff, selected_barrier, config) for name, file in test_files.items()}
exporter.save(
    'event_summary', event_summary([prepared_train, *prepared_tests.values()]),
    'Stacked bars of class shares by file', 'Events, class balance, uniqueness and costs',
)
exporter.save(
    'class_balance_by_stock',
    prepared_train.events.groupby('stock_id')['label'].apply(
        lambda labels: pd.Series(compute_class_shares(labels.to_numpy()))
    ).unstack().reset_index(),
    'Stacked bars of class shares by stock',
    'Training file class balance by stock',
)
exporter.save(
    'feature_descriptive_stats', describe_columns(prepared_train.features[BASE_COLUMNS + DEEP_OFI_COLUMNS]),
    'Table of mean and standard deviation for the data section', 'Training features before scaling',
)

## 4. Model inputs and models

- **Scaling:** `StandardScaler` per stock, fit on each fold's training ticks only. Stock one-hot appended unscaled.
- **Inputs:** tabular models get the event tick; sequence models get the 100 ticks ending at the event. Batches are built on the GPU from one copy of each file.
- **Loss:** uniqueness-weighted cross-entropy. `cost_sensitive` subtracts `cost_loss_weight × E[net] × 1e4`, with `E[net] = p_up(r − c) + p_down(−r − c)`. Returns are used in the loss only, never as inputs.

| Model | Input | Role |
|---|---|---|
| Logistic regression | event tick | Linear baseline |
| MLP small (128-64) | event tick | Non-linear |
| MLP large (256-128-64) | event tick | More neurons and layers |
| LSTM (2 layers) | 100-tick window | Uses recent history |
| CNN+LSTM | 100-tick window | Local patterns + history |

**Scaling and feature set**, chosen in an earlier run on mean walk-forward validation loss (LR, MLP small, LSTM, default settings, 4 folds). Class-prior loss ≈ 1.07.

| Option | LR | MLP small | LSTM |
|---|---|---|---|
| Global scaling, base features | 1.0310 | 1.0236 | 1.0222 |
| Per-stock scaling, base features | 1.0212 | 1.0186 | 1.0191 |
| Per-stock + multi-level OFI | 1.0212 | 1.0172 | 1.0231 |
| **Per-stock + multi-level OFI + handcrafted** | **1.0087** | **1.0071** | 1.0206 |

In [ ]:
class FeatureScaler:
    """Standard scaling, either across all stocks or separately for each stock."""

    def __init__(self, mode):
        if mode not in SCALING_MODES:
            raise ValueError(f'Unknown scaling mode: {mode}')
        self.mode = mode
        self.scalers = {}

    def _groups(self, stock_ids):
        if self.mode == 'global':
            return {-1: np.ones(len(stock_ids), dtype=bool)}
        return {int(stock_id): stock_ids == stock_id for stock_id in np.unique(stock_ids)}

    def fit(self, values, stock_ids):
        self.scalers = {key: StandardScaler().fit(values[mask]) for key, mask in self._groups(stock_ids).items()}
        return self

    def transform(self, values, stock_ids):
        scaled = np.empty(values.shape, dtype=np.float32)
        for key, mask in self._groups(stock_ids).items():
            scaled[mask] = self.scalers[key].transform(values[mask])
        return scaled


@dataclass(frozen=True, eq=False)
class PreparedSplit:
    """Model input for one split. Tabular models use `features`, sequence models use `windows()`."""
    name: str
    tick_features: np.ndarray  # every tick of the file, including rows outside this split
    event_positions: np.ndarray  # row of each event in tick_features
    event_ticks: np.ndarray
    event_ends: np.ndarray
    stock_ids: np.ndarray
    labels: np.ndarray  # -1 down, 0 time-out, 1 up
    weights: np.ndarray
    gross_returns: np.ndarray
    costs: np.ndarray
    feature_names: list
    lookback: int

    @property
    def features(self):
        return self.tick_features[self.event_positions]

    @property
    def class_indices(self):
        """Labels as 0 down, 1 time-out, 2 up, for torch loss functions."""
        return self.labels + 1

    def windows(self, indices=None):
        """Windows ending at each event, shape (n_selected, lookback, n_features)."""
        positions = self.event_positions if indices is None else self.event_positions[indices]
        offsets = np.arange(-self.lookback + 1, 1)
        return self.tick_features[positions[:, None] + offsets]


@dataclass(frozen=True, eq=False)
class FoldData:
    train: PreparedSplit
    validation: PreparedSplit
    scaler: FeatureScaler
    columns: list


def build_tick_matrix(prepared, columns, scaler):
    scaled = scaler.transform(prepared.features[columns].to_numpy(), prepared.stock_ids)
    matrix = np.hstack([scaled, np.eye(N_STOCKS, dtype=np.float32)[prepared.stock_ids]])
    matrix.setflags(write=False)  # splits share this array
    return matrix


def make_split(name, matrix, prepared, events, columns, lookback):
    positions = prepared.features.index.get_indexer(events.index)
    if (positions < 0).any() or (len(positions) > 0 and positions.min() < lookback - 1):
        raise ValueError(f'{name}: some events have no full lookback window')
    return PreparedSplit(
        name=name,
        tick_features=matrix,
        event_positions=positions,
        event_ticks=events.index.to_numpy(),
        event_ends=events['t1'].to_numpy(),
        stock_ids=events['stock_id'].to_numpy(),
        labels=events['label'].to_numpy(),
        weights=events['weight'].to_numpy(),
        gross_returns=events['gross_return'].to_numpy(),
        costs=events['cost'].to_numpy(),
        feature_names=[*columns, *STOCK_COLUMNS],
        lookback=lookback,
    )


def build_fold_data(prepared, fold, columns, scaling, lookback):
    """Purged split: train labels finish before the cut, and validation starts after the gap."""
    events = prepared.events
    t0 = events.index.to_numpy()
    t1 = events['t1'].to_numpy()
    in_train = np.zeros(len(events), dtype=bool)
    in_validation = np.zeros(len(events), dtype=bool)
    for stock_id, stock_fold in fold.items():
        in_stock = events['stock_id'].to_numpy() == stock_id
        in_train |= in_stock & (t1 < stock_fold.train_end)
        in_validation |= in_stock & (t0 >= stock_fold.validation_start) & (t0 < stock_fold.validation_end)

    train_ticks = in_train_period(prepared.features.index.to_numpy(), fold)
    scaler = FeatureScaler(scaling).fit(prepared.features[columns].to_numpy()[train_ticks], prepared.stock_ids[train_ticks])
    matrix = build_tick_matrix(prepared, columns, scaler)
    return FoldData(
        make_split('train', matrix, prepared, events[in_train], columns, lookback),
        make_split('validation', matrix, prepared, events[in_validation], columns, lookback),
        scaler,
        columns,
    )


def build_test_split(prepared, fold_data, lookback):
    """Applies what was learned on training data. Learns nothing from the test file."""
    matrix = build_tick_matrix(prepared, fold_data.columns, fold_data.scaler)
    return make_split(prepared.name, matrix, prepared, prepared.events, fold_data.columns, lookback)


_DEVICE_COPIES = {}


def on_device(array, device):
    """One device copy per array, shared by every batch that uses it."""
    key = (id(array), str(device))
    if key not in _DEVICE_COPIES:
        _DEVICE_COPIES[key] = (array, torch.tensor(array, device=device))
    return _DEVICE_COPIES[key][1]


def clear_device_copies():
    _DEVICE_COPIES.clear()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class EventBatches:
    """Mini-batches of one split, built on the device. Shuffles when given a seed."""

    def __init__(self, split, sequence, batch_size, device, seed=None):
        self.sequence = sequence
        self.batch_size = batch_size
        self.device = device
        self.rng = None if seed is None else np.random.default_rng(seed)
        self.ticks = on_device(split.tick_features, device)
        self.positions = torch.as_tensor(split.event_positions, dtype=torch.long, device=device)
        self.offsets = torch.arange(-split.lookback + 1, 1, device=device)
        self.classes = torch.as_tensor(split.class_indices, dtype=torch.long, device=device)
        self.weights = torch.as_tensor(split.weights, dtype=torch.float32, device=device)
        self.returns = torch.as_tensor(split.gross_returns, dtype=torch.float32, device=device)
        self.costs = torch.as_tensor(split.costs, dtype=torch.float32, device=device)

    def __iter__(self):
        n_events = len(self.classes)
        order = np.arange(n_events) if self.rng is None else self.rng.permutation(n_events)
        for start in range(0, n_events, self.batch_size):
            batch = order[start:start + self.batch_size]
            if self.rng is not None and len(batch) == 1:
                continue  # batch norm cannot train on a single event
            index = torch.as_tensor(batch, device=self.device)
            positions = self.positions[index]
            inputs = self.ticks[positions[:, None] + self.offsets] if self.sequence else self.ticks[positions]
            yield inputs, self.classes[index], self.weights[index], self.returns[index], self.costs[index]

In [ ]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, NUM_CLASSES)

    def forward(self, inputs):
        return self.linear(inputs)


class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers += [nn.Linear(previous_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, NUM_CLASSES))
        self.net = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.net(inputs)


class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        self.output = nn.Linear(hidden_dim, NUM_CLASSES)

    def forward(self, inputs):
        # inputs: (batch, lookback, features)
        sequence, _ = self.lstm(inputs)
        return self.output(sequence[:, -1, :])


class CNNLSTMModel(nn.Module):
    def __init__(self, input_dim, cnn_channels, kernel_size, lstm_hidden_dim, dropout):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, cnn_channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(input_size=cnn_channels, hidden_size=lstm_hidden_dim, batch_first=True)
        self.output = nn.Linear(lstm_hidden_dim, NUM_CLASSES)

    def forward(self, inputs):
        # Conv1d expects (batch, features, time), and the LSTM expects (batch, time, features)
        convolved = self.cnn(inputs.transpose(1, 2)).transpose(1, 2)
        sequence, _ = self.lstm(convolved)
        return self.output(sequence[:, -1, :])


def suggest_common(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 3e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [128, 256, 512]),
    }


def suggest_mlp(trial):
    return {**suggest_common(trial), 'dropout': trial.suggest_float('dropout', 0.0, 0.5)}


def suggest_lstm(trial):
    return {**suggest_mlp(trial), 'lstm_hidden_dim': trial.suggest_categorical('lstm_hidden_dim', [16, 32, 64])}


def suggest_cnn_lstm(trial):
    return {
        **suggest_mlp(trial),
        'lstm_hidden_dim': trial.suggest_categorical('lstm_hidden_dim', [16, 32, 64]),
        'cnn_out_channels': trial.suggest_categorical('cnn_out_channels', [16, 32]),
    }


@dataclass(frozen=True)
class ModelSpec:
    name: str
    sequence: bool  # True uses lookback windows, False uses the event tick only
    build: Callable  # (input_dim, hyperparameters) -> nn.Module
    suggest: Callable  # (optuna trial) -> settings to try


# Listed from simplest to most complex, which the model choice in section 5 relies on.
MODEL_SPECS = [
    ModelSpec('logistic_regression', False, lambda dim, hp: LogisticRegressionModel(dim), suggest_common),
    ModelSpec('mlp_small', False, lambda dim, hp: MLPModel(dim, hp.mlp_small_dims, hp.dropout), suggest_mlp),
    ModelSpec('mlp_large', False, lambda dim, hp: MLPModel(dim, hp.mlp_large_dims, hp.dropout), suggest_mlp),
    ModelSpec('lstm', True, lambda dim, hp: LSTMModel(dim, hp.lstm_hidden_dim, hp.lstm_layers, hp.dropout), suggest_lstm),
    ModelSpec('cnn_lstm', True, lambda dim, hp: CNNLSTMModel(
        dim, hp.cnn_out_channels, hp.cnn_kernel_size, hp.lstm_hidden_dim, hp.dropout
    ), suggest_cnn_lstm),
]


def event_losses(logits, classes, returns, costs, hyperparameters):
    """Loss per event. The cost-sensitive version adds the expected trading loss, in basis points."""
    cross_entropy = F.cross_entropy(logits, classes, reduction='none')
    if hyperparameters.objective == 'cross_entropy':
        return cross_entropy
    probabilities = F.softmax(logits, dim=1)
    expected_net = probabilities[:, 2] * (returns - costs) + probabilities[:, 0] * (-returns - costs)
    return cross_entropy - hyperparameters.cost_loss_weight * expected_net * 1e4


def run_epoch(model, batches, hyperparameters, optimizer=None):
    """One pass over the data. Trains if given an optimizer. Returns the weighted loss."""
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_weight = 0.0
    with torch.set_grad_enabled(training):
        for inputs, classes, weights, returns, costs in batches:
            losses = event_losses(model(inputs), classes, returns, costs, hyperparameters)
            batch_loss = (losses * weights).sum()
            batch_weight = weights.sum()
            if training:
                optimizer.zero_grad()
                (batch_loss / batch_weight).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), hyperparameters.gradient_clip)
                optimizer.step()
            total_loss += batch_loss.item()
            total_weight += batch_weight.item()
    return total_loss / total_weight


def fit_model(model, train_batches, validation_batches, hyperparameters, device, on_epoch=None):
    """Train with learning-rate decay and early stopping, restore the best epoch, return the history."""
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=hyperparameters.learning_rate, weight_decay=hyperparameters.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=hyperparameters.lr_decay_factor,
        patience=hyperparameters.lr_decay_patience, min_lr=hyperparameters.min_learning_rate,
    )
    history = {'train_loss': [], 'validation_loss': [], 'learning_rate': [], 'best_epoch': 0}
    best_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    for epoch in range(1, hyperparameters.max_epochs + 1):
        history['learning_rate'].append(optimizer.param_groups[0]['lr'])
        train_loss = run_epoch(model, train_batches, hyperparameters, optimizer)
        validation_loss = run_epoch(model, validation_batches, hyperparameters)
        scheduler.step(validation_loss)
        history['train_loss'].append(train_loss)
        history['validation_loss'].append(validation_loss)
        if validation_loss < best_loss:
            best_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            history['best_epoch'] = epoch
        if on_epoch is not None:
            on_epoch(epoch, train_loss, validation_loss)
        if epoch - history['best_epoch'] >= hyperparameters.patience:
            break
    model.load_state_dict(best_state)
    return history


def train_new_model(spec, fold_data, hyperparameters, seed, device, on_epoch=None):
    torch.manual_seed(seed)
    model = spec.build(len(fold_data.train.feature_names), hyperparameters)
    history = fit_model(
        model,
        EventBatches(fold_data.train, spec.sequence, hyperparameters.batch_size, device, seed=seed),
        EventBatches(fold_data.validation, spec.sequence, hyperparameters.batch_size, device),
        hyperparameters, device, on_epoch,
    )
    return model, history


def predict_probabilities(model, split, sequence, batch_size, device):
    batches = EventBatches(split, sequence, batch_size, device)
    model.eval()
    with torch.no_grad():
        probabilities = [F.softmax(model(inputs), dim=1).cpu() for inputs, *_ in batches]
    return torch.cat(probabilities).numpy()


def score_split(split, predicted):
    """Classification scores plus per-event returns before and after crossing the spread.

    Up predictions go long, down predictions go short, and time-out predictions do nothing.
    Events overlap, so this is a rough check on costs and not a backtest.
    """
    true_classes = split.class_indices
    traded = predicted != BarrierLabel.TIMEOUT + 1
    gross = (predicted[traded] - 1) * split.gross_returns[traded]
    any_trades = bool(traded.any())
    return {
        'accuracy': accuracy_score(true_classes, predicted),
        'weighted_accuracy': accuracy_score(true_classes, predicted, sample_weight=split.weights),
        'macro_f1': f1_score(true_classes, predicted, average='macro', zero_division=0),
        'trade_share': float(traded.mean()),
        'direction_precision': float((predicted[traded] == true_classes[traded]).mean()) if any_trades else np.nan,
        'gross_bps': float(gross.mean() * 1e4) if any_trades else np.nan,
        'net_bps': float((gross - split.costs[traded]).mean() * 1e4) if any_trades else np.nan,
    }

## 5. Pipeline: tune, compare, select, calibrate

`run_setup` applies the same steps to every setup:

1. Build the 4 purged walk-forward folds.
2. Tune each model with Optuna (TPE, 20 trials, median pruning) on mean validation loss over folds 2-3.
3. Train every model on every fold with 3 seeds.
4. Select the simplest model whose mean macro F1 is within 1 seed SD of the best.
5. Final models = fold 4 models; the 3 seeds are averaged into an ensemble.
6. Calibrate with temperature scaling on fold 4 validation (does not change the predicted class).
7. Score against baselines: majority class, always up, always down.

`net_bps` is the mean per-trade return after one spread crossing. Events overlap, so this is not a backtest.

In [ ]:
PIPELINE_CODE = [
    compute_mid_price, build_segments, compute_folds, FractionalDifferentiator, build_segment_features,
    build_tick_features, compute_rolling_volatility, detect_cusum_events, filter_eligible_events, sample_events,
    barrier_widths, label_events, compute_uniqueness, prepare_file, FeatureScaler, build_tick_matrix, make_split, build_fold_data,
]
MODEL_CODE = [
    LogisticRegressionModel, MLPModel, LSTMModel, CNNLSTMModel, EventBatches, event_losses, run_epoch, fit_model,
    train_new_model,
]


def data_key(config, columns, scaling, d, barrier):
    settings = {'config': asdict(config), 'columns': list(columns), 'scaling': scaling, 'd': d, 'barrier': barrier}
    return fingerprint(settings, PIPELINE_CODE)


@dataclass(frozen=True, eq=False)
class TrainedModel:
    spec: ModelSpec
    seed: int
    fold: int
    model: nn.Module
    hyperparameters: ModelConfig
    history: dict


@dataclass(frozen=True, eq=False)
class SetupResult:
    name: str
    config: PipelineConfig
    prepared_tests: dict
    fold_datas: list
    data_key: str
    tuning_results: dict
    walk_forward_scores: pd.DataFrame
    selected_model: str
    highest_scoring_model: str
    final_runs: dict
    temperatures: dict
    validation_probabilities: dict
    validation_predictions: list


def tune_hyperparameters(spec, fold_datas, base, device):
    """Mean validation loss over the tuning folds. Weak trials stop after the first fold."""
    input_dim = len(fold_datas[0].train.feature_names)
    seed = base.seeds[0]

    def objective(trial):
        candidate = replace(base, **spec.suggest(trial))
        losses = []
        for step, fold_data in enumerate(fold_datas):
            _, history = train_new_model(spec, fold_data, candidate, seed, device)
            losses.append(min(history['validation_loss']))
            trial.report(float(np.mean(losses)), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(losses))

    # A different search for each model, repeatable because crc32 is stable between sessions
    sampler = optuna.samplers.TPESampler(
        seed=seed + zlib.crc32(spec.name.encode()) % 100000, n_startup_trials=base.tuning_startup_trials,
    )
    pruner = optuna.pruners.MedianPruner(n_startup_trials=base.tuning_startup_trials)
    study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=base.tuning_trials)
    pruned = sum(trial.state == optuna.trial.TrialState.PRUNED for trial in study.trials)
    return {'best_params': study.best_params, 'best_validation_loss': study.best_value, 'pruned_trials': pruned}


def make_epoch_logger(run, name):
    def log_epoch(epoch, train_loss, validation_loss):
        if epoch == 1 or epoch % 5 == 0:
            print(f'{name} | Epoch {epoch} | Train loss: {train_loss:.5f} | Validation loss: {validation_loss:.5f}')
        if run is not None:
            run.log({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss})
    return log_epoch


def train_or_load(spec, fold_data, hyperparameters, seed, fold_index, key, is_final):
    """Reuse saved weights when settings and code match. Final models are logged to W&B."""
    path = context.weights_dir / f'{spec.name}_{key}_fold{fold_index + 1}_seed{seed}.pt'
    if path.exists() and not RETRAIN_MODELS:
        checkpoint = torch.load(path, map_location=context.device, weights_only=True)
        model = spec.build(len(fold_data.train.feature_names), hyperparameters).to(context.device)
        model.load_state_dict(checkpoint['state_dict'])
        return TrainedModel(spec, seed, fold_index, model, hyperparameters, checkpoint['history'])

    name = f'{spec.name}_seed{seed}'
    run = None
    if is_final and context.wandb_enabled:
        run = wandb.init(project=WANDB_PROJECT, name=name, config=asdict(hyperparameters))
    on_epoch = make_epoch_logger(run, name) if is_final else None
    model, history = train_new_model(spec, fold_data, hyperparameters, seed, context.device, on_epoch)
    if run is not None:
        run.finish()
    torch.save({'state_dict': model.state_dict(), 'history': history}, path)
    return TrainedModel(spec, seed, fold_index, model, hyperparameters, history)


def predict_trained(trained, split):
    return predict_probabilities(trained.model, split, trained.spec.sequence, trained.hyperparameters.batch_size, context.device)


def ensemble_probabilities(runs, split):
    return np.mean([predict_trained(trained, split) for trained in runs], axis=0)


def baseline_predictions(train_split, split):
    """Predictions every model must beat: the most common training class, always up, and always down."""
    majority = int(np.bincount(train_split.class_indices).argmax())
    n_events = len(split.labels)
    return {
        'majority_class': np.full(n_events, majority),
        'always_up': np.full(n_events, 2),
        'always_down': np.full(n_events, 0),
    }


def summarize_scores(scores):
    """Mean and spread across seeds, after averaging over folds or files within each seed."""
    seed_means = scores.groupby(['model', 'seed'], sort=False)[SCORE_COLUMNS].mean()
    summary = seed_means.groupby('model', sort=False).agg(['mean', 'std'])
    summary.columns = [f'{metric}_{statistic}' for metric, statistic in summary.columns]
    return summary


def choose_simplest_good_model(summary, model_names):
    """The first model, from simplest up, within one seed std of the best mean macro F1."""
    models = summary.loc[model_names]
    best = models['macro_f1_mean'].idxmax()
    cutoff = models.loc[best, 'macro_f1_mean'] - models.loc[best, 'macro_f1_std']
    for name in model_names:
        if models.loc[name, 'macro_f1_mean'] >= cutoff:
            return name, best
    return best, best


def apply_temperature(probabilities, temperature):
    scaled = np.power(np.clip(probabilities, 1e-12, 1.0), 1.0 / temperature)
    return scaled / scaled.sum(axis=1, keepdims=True)


def weighted_log_loss(probabilities, split):
    event_probabilities = probabilities[np.arange(len(probabilities)), split.class_indices]
    return float(-(np.log(np.clip(event_probabilities, 1e-12, 1.0)) * split.weights).sum() / split.weights.sum())


def fit_temperature(probabilities, split):
    candidates = np.round(np.arange(0.5, 3.01, 0.05), 2)
    losses = [weighted_log_loss(apply_temperature(probabilities, value), split) for value in candidates]
    return float(candidates[int(np.argmin(losses))])


def build_prediction_rows(final_runs, train_split, splits, probabilities_by_split):
    """One row per model, seed and split, plus ensembles and baselines."""
    rows = []
    for split_name, split in splits.items():
        for baseline, predicted in baseline_predictions(train_split, split).items():
            rows.append({'model': baseline, 'seed': 0, 'split': split_name, 'predicted': predicted})
        for model_name, runs in final_runs.items():
            for trained in runs:
                rows.append({
                    'model': model_name, 'seed': trained.seed, 'split': split_name,
                    'predicted': predict_trained(trained, split).argmax(axis=1),
                })
            rows.append({
                'model': f'{model_name}_ensemble', 'seed': 0, 'split': split_name,
                'predicted': probabilities_by_split[split_name][model_name].argmax(axis=1),
            })
    return rows


def run_setup(name, prepared_train, prepared_tests, setup_config, setup_model_config, specs):
    """Tune, compare, choose, finalize and calibrate one setup."""
    clear_device_copies()
    setup_folds = compute_folds(train_file.segments, setup_config)
    fold_datas = [
        build_fold_data(prepared_train, fold, selected_columns, selected_scaling, setup_config.lookback)
        for fold in setup_folds
    ]
    key = data_key(setup_config, selected_columns, selected_scaling, selected_d, selected_barrier)
    base_settings = {k: v for k, v in asdict(setup_model_config).items() if k != 'seeds'}
    tuning_folds = [fold_datas[index] for index in setup_model_config.tuning_folds]

    tuning_results = {}
    walk_forward_runs = {}
    for spec in specs:
        tuning_key = fingerprint(
            {'data': key, 'model': spec.name, 'hyperparameters': base_settings},
            [*MODEL_CODE, tune_hyperparameters, spec.build, spec.suggest],
        )
        tuning_results[spec.name] = cached_json(
            context.tuning_dir / f'{name}_{spec.name}_{tuning_key}.json',
            lambda: tune_hyperparameters(spec, tuning_folds, setup_model_config, context.device),
        )
        hyperparameters = replace(setup_model_config, **tuning_results[spec.name]['best_params'])
        weights_key = fingerprint({'data': key, 'hyperparameters': asdict(hyperparameters)}, [*MODEL_CODE, spec.build])
        walk_forward_runs[spec.name] = [
            [
                train_or_load(spec, fold_data, hyperparameters, seed, fold_index, weights_key, fold_index == len(fold_datas) - 1)
                for seed in setup_model_config.seeds
            ]
            for fold_index, fold_data in enumerate(fold_datas)
        ]
        print(f'{name} | {spec.name}: tuned and trained')

    score_rows = []
    for fold_index, fold_data in enumerate(fold_datas):
        split = fold_data.validation
        for baseline, predicted in baseline_predictions(fold_data.train, split).items():
            score_rows.append({'model': baseline, 'seed': 0, 'fold': fold_index + 1, **score_split(split, predicted)})
        for model_name, by_fold in walk_forward_runs.items():
            for trained in by_fold[fold_index]:
                predicted = predict_trained(trained, split).argmax(axis=1)
                score_rows.append({'model': model_name, 'seed': trained.seed, 'fold': fold_index + 1, **score_split(split, predicted)})
    walk_forward_scores = pd.DataFrame(score_rows)
    selected_model, highest_scoring_model = choose_simplest_good_model(
        summarize_scores(walk_forward_scores), [spec.name for spec in specs],
    )

    final_runs = {model_name: by_fold[-1] for model_name, by_fold in walk_forward_runs.items()}
    final_fold = fold_datas[-1]
    raw_probabilities = {model_name: ensemble_probabilities(runs, final_fold.validation) for model_name, runs in final_runs.items()}
    temperatures = {model_name: fit_temperature(values, final_fold.validation) for model_name, values in raw_probabilities.items()}
    validation_probabilities = {
        model_name: apply_temperature(values, temperatures[model_name]) for model_name, values in raw_probabilities.items()
    }
    validation_predictions = build_prediction_rows(
        final_runs, final_fold.train, {'validation': final_fold.validation}, {'validation': validation_probabilities},
    )
    return SetupResult(
        name, setup_config, prepared_tests, fold_datas, key, tuning_results, walk_forward_scores,
        selected_model, highest_scoring_model, final_runs, temperatures, validation_probabilities, validation_predictions,
    )

In [ ]:
TRAIN_COLOR = '#2a78d6'
VALIDATION_COLOR = '#eb6834'
TEXT_COLOR = '#52514e'
REFERENCE_COLOR = '#8a8982'
SURFACE_COLOR = '#fcfcfb'
SERIES_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#6250d6']

plt.rcParams.update({
    'figure.facecolor': SURFACE_COLOR, 'axes.facecolor': SURFACE_COLOR,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#c9c8c1',
    'axes.labelcolor': TEXT_COLOR, 'axes.titlesize': 11, 'axes.titlelocation': 'left',
    'axes.axisbelow': True, 'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8,
    'xtick.color': TEXT_COLOR, 'ytick.color': TEXT_COLOR, 'lines.linewidth': 2,
})


def plot_learning_curves(final_runs, title):
    """One panel per model on a shared scale, with one line per seed."""
    fig, axes = plt.subplots(1, len(final_runs), figsize=(3 * len(final_runs), 3.5), sharey=True, squeeze=False)
    for ax, (model_name, runs) in zip(axes[0], final_runs.items()):
        for index, trained in enumerate(runs):
            history = trained.history
            epochs = range(1, len(history['validation_loss']) + 1)
            labels = ('Train', 'Validation', 'Best epoch') if index == 0 else (None, None, None)
            ax.plot(epochs, history['train_loss'], color=TRAIN_COLOR, alpha=0.8, label=labels[0])
            ax.plot(epochs, history['validation_loss'], color=VALIDATION_COLOR, alpha=0.8, label=labels[1])
            ax.scatter(
                history['best_epoch'], history['validation_loss'][history['best_epoch'] - 1], s=64, zorder=3,
                color=VALIDATION_COLOR, edgecolors=SURFACE_COLOR, linewidths=2, label=labels[2],
            )
        ax.set_title(model_name)
        ax.set_xlabel('Epoch')
    axes[0][0].set_ylabel('Weighted loss')
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False)
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    plt.show()


def plot_model_scores(summary, metric, title):
    """Mean across seeds with the spread as error bars, against the most-common-class baseline."""
    baseline = summary.loc['majority_class', f'{metric}_mean']
    models = summary.drop(index=[name for name in summary.index if name in BASELINES or name.endswith('_ensemble')])
    means = models[f'{metric}_mean'].to_numpy()
    spreads = models[f'{metric}_std'].fillna(0).to_numpy()
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(models) + 1.8))
    ax.barh(models.index.to_list(), means, xerr=spreads, height=0.6, color=TRAIN_COLOR, error_kw={'ecolor': TEXT_COLOR, 'capsize': 3})
    for position, value in enumerate(means):
        ax.text(0.005, position, f'{value:.3f}', va='center', color=SURFACE_COLOR)
    ax.axvline(baseline, color=REFERENCE_COLOR, linestyle='--', linewidth=1.5)
    ax.text(baseline, 1.01, f'Most common class {baseline:.3f}', transform=ax.get_xaxis_transform(), ha='center', va='bottom', color=TEXT_COLOR)
    ax.invert_yaxis()
    ax.grid(axis='y', visible=False)
    ax.set_xlim(0, max((means + spreads).max(), baseline) * 1.1)
    ax.set_xlabel(metric)
    ax.set_title(title, pad=24)
    fig.tight_layout()
    plt.show()


def plot_confusion_matrices(matrices, title):
    """Share of each true class given each prediction."""
    fig, axes = plt.subplots(1, len(matrices), figsize=(3 * len(matrices), 3.6), sharey=True, squeeze=False)
    for ax, (model_name, matrix) in zip(axes[0], matrices.items()):
        ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1)
        for row in range(NUM_CLASSES):
            for column in range(NUM_CLASSES):
                color = SURFACE_COLOR if matrix[row, column] > 0.5 else TEXT_COLOR
                ax.text(column, row, f'{matrix[row, column]:.2f}', ha='center', va='center', color=color)
        ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=30)
        ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
        ax.grid(False)
        ax.set_title(model_name.replace('_ensemble', ''))
        ax.set_xlabel('Predicted')
    axes[0][0].set_ylabel('True')
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()


def plot_cumulative_returns(table, model_names, title):
    """Cumulative per-event return for chosen ensembles and the always-up and always-down baselines."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
    for ax, kind in zip(axes, ('gross', 'net')):
        for color, model_name in zip(SERIES_COLORS, [*model_names, 'always_up', 'always_down']):
            column = f'{model_name}_{kind}_bps'
            style = '--' if model_name in BASELINES else '-'
            ax.plot(table['event'], table[column], color=color, linestyle=style, label=model_name.replace('_ensemble', ''))
        ax.axhline(0, color=REFERENCE_COLOR, linewidth=1)
        ax.set_title(f'{kind.capitalize()} cumulative return (bps)')
        ax.set_xlabel('Event, in time order within each stock')
    axes[0].legend(frameon=False)
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()


def build_score_table(predictions, splits):
    return pd.DataFrame([
        {'model': row['model'], 'seed': row['seed'], 'split': row['split'], **score_split(splits[row['split']], row['predicted'])}
        for row in predictions
    ])


def pool_predictions(predictions, splits, model_name):
    rows = [row for row in predictions if row['model'] == model_name]
    true_classes = np.concatenate([splits[row['split']].class_indices for row in rows])
    predicted = np.concatenate([row['predicted'] for row in rows])
    return true_classes, predicted


def build_per_class_table(predictions, splits):
    """Precision, recall and F1 for each class, pooled over seeds and files."""
    rows = []
    for model_name in dict.fromkeys(row['model'] for row in predictions):
        true_classes, predicted = pool_predictions(predictions, splits, model_name)
        precision, recall, f1, support = precision_recall_fscore_support(
            true_classes, predicted, labels=list(range(NUM_CLASSES)), zero_division=0
        )
        for index, class_name in enumerate(CLASS_NAMES):
            rows.append({
                'model': model_name, 'class': class_name, 'precision': precision[index], 'recall': recall[index],
                'f1': f1[index], 'support': int(support[index]), 'predicted_share': float(np.mean(predicted == index)),
                'true_share': support[index] / len(true_classes),
            })
    return pd.DataFrame(rows)


def build_confusion_outputs(predictions, splits):
    rows = []
    matrices = {}
    for model_name in dict.fromkeys(row['model'] for row in predictions):
        true_classes, predicted = pool_predictions(predictions, splits, model_name)
        labels = list(range(NUM_CLASSES))
        counts = confusion_matrix(true_classes, predicted, labels=labels)
        rates = confusion_matrix(true_classes, predicted, labels=labels, normalize='true')
        if model_name.endswith('_ensemble'):
            matrices[model_name] = rates
        for true_index, true_name in enumerate(CLASS_NAMES):
            for predicted_index, predicted_name in enumerate(CLASS_NAMES):
                rows.append({
                    'model': model_name, 'true_class': true_name, 'predicted_class': predicted_name,
                    'count': counts[true_index, predicted_index], 'share_of_true_class': rates[true_index, predicted_index],
                })
    return pd.DataFrame(rows), matrices


def build_bootstrap_table(predictions, splits, reference='logistic_regression', resamples=1000):
    """Rough 95% intervals for accuracy and its gap to logistic regression. Events are treated as independent."""
    correct = {}
    for model_name in dict.fromkeys(row['model'] for row in predictions):
        parts = []
        for split_name, split in splits.items():
            runs = [
                row['predicted'] == split.class_indices
                for row in predictions if row['model'] == model_name and row['split'] == split_name
            ]
            parts.append(np.mean(runs, axis=0))
        correct[model_name] = np.concatenate(parts)
    n_events = len(correct[reference])
    samples = np.random.default_rng(0).integers(0, n_events, size=(resamples, n_events))
    reference_draws = correct[reference][samples].mean(axis=1)
    rows = []
    for model_name, values in correct.items():
        draws = values[samples].mean(axis=1)
        gaps = draws - reference_draws
        rows.append({
            'model': model_name, 'accuracy': values.mean(),
            'accuracy_low': np.percentile(draws, 2.5), 'accuracy_high': np.percentile(draws, 97.5),
            'gap_to_logistic': values.mean() - correct[reference].mean(),
            'gap_low': np.percentile(gaps, 2.5), 'gap_high': np.percentile(gaps, 97.5),
        })
    return pd.DataFrame(rows)


def build_cumulative_returns(predictions, split_name, split):
    """Cumulative per-event return, averaged over seeds. Not a backtest: events overlap and fills are assumed."""
    columns = {'event': np.arange(len(split.labels))}
    for model_name in dict.fromkeys(row['model'] for row in predictions):
        runs = [row['predicted'] for row in predictions if row['model'] == model_name and row['split'] == split_name]
        positions = np.array(runs) - 1
        gross = positions * split.gross_returns
        net = gross - (positions != 0) * split.costs
        columns[f'{model_name}_gross_bps'] = np.cumsum(gross, axis=1).mean(axis=0) * 1e4
        columns[f'{model_name}_net_bps'] = np.cumsum(net, axis=1).mean(axis=0) * 1e4
    return pd.DataFrame(columns)


def report_results(label, splits, predictions, highlight_models):
    """Every results table and chart, used the same way for validation and test."""
    scores = build_score_table(predictions, splits)
    summary = summarize_scores(scores)
    exporter.save(f'{label}_scores_by_seed', scores, 'Source table for your own pivots', show=False)
    exporter.save(
        f'{label}_summary', summary.reset_index(),
        'Bars of macro_f1_mean and net_bps_mean by model, with std as error bars',
        f'{label}: mean and spread across seeds. Ensembles and baselines have one row each, so their std is blank.',
    )
    exporter.save(f'{label}_per_class', build_per_class_table(predictions, splits), 'Clustered bars of precision and recall by class', f'{label}: each class')
    confusion_table, matrices = build_confusion_outputs(predictions, splits)
    exporter.save(f'{label}_confusion', confusion_table, 'Pivot share_of_true_class into a heatmap', show=False)
    exporter.save(f'{label}_accuracy_intervals', build_bootstrap_table(predictions, splits), 'Bars with error bars from the low and high columns', f'{label}: accuracy intervals')
    plot_model_scores(summary, 'macro_f1', f'{label}: macro F1 by model')
    plot_confusion_matrices(matrices, f'{label}: share of each true class by prediction (ensembles)')
    for split_name, split in splits.items():
        table = build_cumulative_returns(predictions, split_name, split)
        exporter.save(
            f'{label}_cumulative_returns_{split_name}', table,
            'Line chart of gross and net columns by event', show=False,
        )
        plot_cumulative_returns(table, [f'{name}_ensemble' for name in highlight_models], f'{label}, {split_name}: cumulative return')
    return scores


def report_setup(result):
    """The walk-forward comparison, model details, calibration and validation results for one setup."""
    name = result.name
    exporter.save(
        f'{name}_tuning',
        pd.DataFrame([
            {'model': model_name, 'best_validation_loss': values['best_validation_loss'],
             'pruned_trials': values['pruned_trials'], **values['best_params']}
            for model_name, values in result.tuning_results.items()
        ]),
        'Table for the methods section', f'{name}: best settings found for each model',
    )
    exporter.save(f'{name}_walk_forward_by_fold', result.walk_forward_scores, 'Line chart of macro_f1 by fold and model', show=False)
    summary = summarize_scores(result.walk_forward_scores)
    exporter.save(
        f'{name}_walk_forward_summary', summary.reset_index(),
        'Bars of macro_f1_mean by model, with macro_f1_std as error bars', f'{name}: walk-forward summary',
    )
    print(f'{name}: highest macro F1 is {result.highest_scoring_model}, and the simplicity rule selects {result.selected_model}')
    plot_model_scores(summary, 'macro_f1', f'{name}: walk-forward macro F1 by model')

    exporter.save(
        f'{name}_architectures',
        pd.DataFrame([
            {
                'model': model_name,
                'input': '100-tick window' if runs[0].spec.sequence else 'event tick only',
                'trainable_parameters': sum(parameter.numel() for parameter in runs[0].model.parameters()),
                **{key: getattr(runs[0].hyperparameters, key) for key in ('learning_rate', 'weight_decay', 'batch_size', 'dropout', 'lstm_hidden_dim', 'cnn_out_channels')},
                'mean_epochs_run': np.mean([len(run.history['train_loss']) for run in runs]),
                'mean_best_epoch': np.mean([run.history['best_epoch'] for run in runs]),
                'mean_final_learning_rate': np.mean([run.history['learning_rate'][-1] for run in runs]),
            }
            for model_name, runs in result.final_runs.items()
        ]),
        'Table of model designs and sizes. Dropout, LSTM and CNN columns apply only to models that use them',
        f'{name}: final models',
    )
    exporter.save(
        f'{name}_learning_curves',
        pd.DataFrame([
            {'model': model_name, 'seed': trained.seed, 'epoch': position + 1,
             'train_loss': trained.history['train_loss'][position],
             'validation_loss': trained.history['validation_loss'][position],
             'learning_rate': trained.history['learning_rate'][position],
             'is_best_epoch': position + 1 == trained.history['best_epoch']}
            for model_name, runs in result.final_runs.items() for trained in runs
            for position in range(len(trained.history['train_loss']))
        ]),
        'Line chart of loss by epoch for each model and seed', show=False,
    )
    plot_learning_curves(result.final_runs, f'{name}: learning curves of the final models')
    exporter.save(
        f'{name}_calibration',
        pd.DataFrame([{'model': model_name, 'temperature': value} for model_name, value in result.temperatures.items()]),
        'Table. Above 1 means the model was over-confident', f'{name}: calibration temperatures',
    )
    validation_split = result.fold_datas[-1].validation
    return report_results(
        f'{name}_validation', {'validation': validation_split}, result.validation_predictions,
        list(dict.fromkeys([result.selected_model, 'logistic_regression'])),
    )

## 6. Run the three setups

- **`main`:** volatility-scaled labels, cross-entropy.
- **`cost_labels`:** in `main` the typical move is smaller than the spread, so barriers are floored at 2 spreads and the holding period is chosen from (50, 100, 200, 400) for 20-35% time-outs on the settings period.
- **`cost_loss`:** `cost_labels` data with the cost-sensitive loss.

Each setup is tuned and selected separately with all 5 models. The cost-aware setups predict a different target, so compare setups on `direction_precision` and `net_bps`, not macro F1.

In [ ]:
main_result = run_setup('main', prepared_train, prepared_tests, config, model_config, MODEL_SPECS)
main_validation_scores = report_setup(main_result)

In [ ]:
def compare_holding_periods(file, volatility, frac_diff, config, fold):
    rows = []
    for holding_period in COST_AWARE_HOLDING_CANDIDATES:
        candidate = replace(config, min_barrier_spreads=COST_AWARE_SPREADS, holding_period=holding_period)
        ticks = settings_period_events(sample_events(file, volatility, frac_diff, candidate), holding_period, fold)
        widths = barrier_widths(volatility, file.relative_spread, selected_barrier, candidate)
        events = label_events(ticks, file.mid_price, widths, file.segments, holding_period)
        moves = np.log(file.mid_price[events['t1'].to_numpy()] / file.mid_price[events.index.to_numpy()])
        rows.append({
            'holding_period': holding_period, 'events': len(events), **compute_class_shares(events['label'].to_numpy()),
            'mean_abs_move_bps': np.abs(moves).mean() * 1e4,
            'mean_cost_bps': file.relative_spread[events.index.to_numpy()].mean() * 1e4,
        })
    return rows


holding_rows = compare_holding_periods(train_file, train_volatility, frac_diff, config, settings_fold)
exporter.save(
    'cost_aware_holding_periods', pd.DataFrame(holding_rows),
    'Line chart of timeout_share and mean_abs_move_bps by holding period',
    'Cost-aware labels by holding period, on the settings period',
)
low, high = config.target_timeout_range
cost_holding = min(holding_rows, key=lambda row: abs(row['timeout_share'] - (low + high) / 2))['holding_period']
cost_config = replace(config, min_barrier_spreads=COST_AWARE_SPREADS, holding_period=cost_holding)
print(f'Selected holding period: {cost_holding} ticks')

cost_prepared_train = prepare_file(train_file, frac_diff, selected_barrier, cost_config)
cost_prepared_tests = {name: prepare_file(file, frac_diff, selected_barrier, cost_config) for name, file in test_files.items()}
exporter.save(
    'cost_aware_event_summary', event_summary([cost_prepared_train, *cost_prepared_tests.values()]),
    'Stacked bars of class shares by file', 'Cost-aware labels: events and class balance',
)

cost_label_result = run_setup('cost_labels', cost_prepared_train, cost_prepared_tests, cost_config, model_config, MODEL_SPECS)
cost_label_validation_scores = report_setup(cost_label_result)

In [ ]:
cost_loss_config = replace(model_config, objective='cost_sensitive')
cost_loss_result = run_setup('cost_loss', cost_prepared_train, cost_prepared_tests, cost_config, cost_loss_config, MODEL_SPECS)
cost_loss_validation_scores = report_setup(cost_loss_result)

In [ ]:
# Setup comparison on fold 4 validation (ensembles and baselines)
SETUPS = {'main': main_result, 'cost_labels': cost_label_result, 'cost_loss': cost_loss_result}
VALIDATION_SCORES = {
    'main': main_validation_scores, 'cost_labels': cost_label_validation_scores, 'cost_loss': cost_loss_validation_scores,
}


def setup_comparison(scores_by_setup):
    rows = []
    for setup_name, scores in scores_by_setup.items():
        summary = summarize_scores(scores)
        keep = [name for name in summary.index if name.endswith('_ensemble') or name in BASELINES]
        for model_name in keep:
            rows.append({
                'setup': setup_name, 'model': model_name.replace('_ensemble', ''),
                **{metric: summary.loc[model_name, f'{metric}_mean'] for metric in SCORE_COLUMNS},
            })
    return pd.DataFrame(rows)


exporter.save(
    'setup_comparison_validation', setup_comparison(VALIDATION_SCORES),
    'Clustered bars of net_bps and direction_precision by setup and model',
    'All setups on validation (ensembles and baselines)',
)

## 7. Test

Runs only with `EVALUATE_TEST = True`; each run is appended to `test_evaluations.csv`. Test files use the fold 4 scaler, models and temperatures, and influence no choice.

In [ ]:
def record_test_evaluation():
    """Add a line to the log and return how many times the test files have been scored."""
    path = context.project_dir / 'test_evaluations.csv'
    row = pd.DataFrame([{
        'time': datetime.now().isoformat(), 'run': context.run_dir.name, 'data_fingerprint': main_result.data_key,
        'selected_model': main_result.selected_model,
    }])
    history = pd.concat([pd.read_csv(path), row]) if path.exists() else row
    history.to_csv(path, index=False)
    return len(history)


def evaluate_on_test(result):
    clear_device_copies()
    final_fold = result.fold_datas[-1]
    test_splits = {name: build_test_split(prepared, final_fold, result.config.lookback) for name, prepared in result.prepared_tests.items()}
    probabilities = {
        split_name: {
            model_name: apply_temperature(ensemble_probabilities(runs, split), result.temperatures[model_name])
            for model_name, runs in result.final_runs.items()
        }
        for split_name, split in test_splits.items()
    }
    predictions = build_prediction_rows(result.final_runs, final_fold.train, test_splits, probabilities)
    scores = report_results(
        f'{result.name}_test', test_splits, predictions,
        list(dict.fromkeys([result.selected_model, 'logistic_regression'])),
    )
    scores.to_csv(context.run_dir / f'{result.name}_test_scores.csv', index=False)
    return {'splits': test_splits, 'probabilities': probabilities, 'scores': scores}


TEST_RESULTS = {}
if EVALUATE_TEST:
    print(f'Test evaluation number {record_test_evaluation()}')
    TEST_RESULTS = {setup_name: evaluate_on_test(result) for setup_name, result in SETUPS.items()}
    exporter.save(
        'setup_comparison_test', setup_comparison({name: values['scores'] for name, values in TEST_RESULTS.items()}),
        'Clustered bars of net_bps and direction_precision by setup and model',
        'All setups on test (ensembles and baselines)',
    )
else:
    print('Test files are locked. Set EVALUATE_TEST = True once the model is chosen.')

## 8. Business scenarios: precision vs recall

Act only when the top class is up/down and calibrated confidence ≥ threshold (0.34-0.90). Thresholds are picked on validation and applied unchanged to test. Each setup's selected model is compared with logistic regression.

| Scenario | User | Rule (≥ 100 trades) |
|---|---|---|
| Must-fill orders | Broker who trades anyway | Max precision with mean up/down recall ≥ 50% |
| Balanced monitoring | Market maker | Max mean up/down F1 |
| Selective trading | Proprietary trader | Max precision |

In [ ]:
THRESHOLDS = np.round(np.arange(0.34, 0.92, 0.02), 2)
MIN_TRADES = 100


def safe_mean(values):
    return float(values.mean()) if len(values) else np.nan


def threshold_rows(setup_name, model_name, group, probabilities, splits):
    true_classes = np.concatenate([split.class_indices for split in splits])
    returns = np.concatenate([split.gross_returns for split in splits])
    costs = np.concatenate([split.costs for split in splits])
    predicted = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    rows = []
    for threshold in THRESHOLDS:
        confident = confidence >= threshold
        traded = confident & (predicted != 1)
        gross = (predicted[traded] - 1) * returns[traded]
        row = {
            'setup': setup_name, 'model': model_name, 'group': group, 'threshold': threshold,
            'trades': int(traded.sum()), 'coverage': float(traded.mean()),
            'direction_precision': safe_mean(predicted[traded] == true_classes[traded]),
            'gross_bps': safe_mean(gross) * 1e4, 'net_bps': safe_mean(gross - costs[traded]) * 1e4,
        }
        recalls = []
        f1_scores = []
        for class_name, class_index in (('down', 0), ('up', 2)):
            chosen = confident & (predicted == class_index)
            actual = true_classes == class_index
            precision = safe_mean(actual[chosen])
            recall = (chosen & actual).sum() / max(actual.sum(), 1)
            row[f'{class_name}_precision'] = precision
            row[f'{class_name}_recall'] = recall
            recalls.append(recall)
            f1_scores.append(0.0 if np.isnan(precision) or precision + recall == 0 else 2 * precision * recall / (precision + recall))
        row['mean_recall'] = float(np.mean(recalls))
        row['mean_f1'] = float(np.mean(f1_scores))
        rows.append(row)
    return rows


def choose_operating_points(table):
    """One validation threshold per scenario."""
    usable = table[table['trades'] >= MIN_TRADES]
    if usable.empty:
        return {}
    must_fill = usable[usable['mean_recall'] >= 0.5]
    must_fill = must_fill if len(must_fill) else usable
    return {
        'Must-fill orders': must_fill.loc[must_fill['direction_precision'].idxmax(), 'threshold'],
        'Balanced monitoring': usable.loc[usable['mean_f1'].idxmax(), 'threshold'],
        'Selective trading': usable.loc[usable['direction_precision'].idxmax(), 'threshold'],
    }


all_threshold_rows = []
scenario_rows = []
for setup_name, result in SETUPS.items():
    validation_split = result.fold_datas[-1].validation
    for model_name in dict.fromkeys([result.selected_model, 'logistic_regression']):
        rows = threshold_rows(setup_name, model_name, 'validation', result.validation_probabilities[model_name], [validation_split])
        if setup_name in TEST_RESULTS:
            test = TEST_RESULTS[setup_name]
            rows += threshold_rows(
                setup_name, model_name, 'test',
                np.vstack([test['probabilities'][name][model_name] for name in test['splits']]),
                list(test['splits'].values()),
            )
        table = pd.DataFrame(rows)
        all_threshold_rows += rows
        for scenario, threshold in choose_operating_points(table[table['group'] == 'validation']).items():
            for _, row in table[table['threshold'] == threshold].iterrows():
                scenario_rows.append({'scenario': scenario, **row.to_dict()})

exporter.save(
    'precision_recall_by_threshold', pd.DataFrame(all_threshold_rows),
    'Scatter of up_precision against up_recall by model. Ignore rows with very few trades', show=False,
)
exporter.save(
    'business_scenarios', pd.DataFrame(scenario_rows),
    'Table for the slides: one row per scenario, setup, model and data group',
    'Operating point for each scenario. Thresholds come from validation.',
)

## 9. Diagnostics

- **Permutation importance:** rise in validation loss when one feature group is shuffled (main setup, all seeds). Shows reliance, not causation.
- **Cumulative returns:** every model per setup, gross and net, on validation.
- **Calibration:** accuracy per confidence band vs the diagonal.
- **Results by spread:** trades in 5 equal spread groups.
- **Example labels:** one down, time-out and up event with its barriers.

In [ ]:
def build_feature_groups(columns):
    level_one = {'ask_price_1', 'bid_price_1', 'ask_size_1', 'bid_size_1'}
    groups = {
        'level_1_prices': ['ask_price_1', 'bid_price_1'],
        'level_1_sizes': ['ask_size_1', 'bid_size_1'],
        'deeper_prices': [column for column in PRICE_COLUMNS if column not in level_one],
        'deeper_sizes': [column for column in BOOK_COLUMNS if '_size_' in column and column not in level_one],
        'stock_id': STOCK_COLUMNS,
        **{column: [column] for column in ENGINEERED_COLUMNS},
    }
    if 'ofi_2' in columns:
        groups['deeper_ofi'] = DEEP_OFI_COLUMNS
    if 'handcrafted_0' in columns:
        groups['handcrafted'] = HANDCRAFTED_COLUMNS
    names = [*columns, *STOCK_COLUMNS]
    return {name: [names.index(column) for column in group] for name, group in groups.items()}


def cross_entropy_on_inputs(model, inputs, split, batch_size):
    device = context.device
    classes = torch.as_tensor(split.class_indices, dtype=torch.long, device=device)
    weights = torch.as_tensor(split.weights, dtype=torch.float32, device=device)
    total = 0.0
    model.eval()
    with torch.no_grad():
        for start in range(0, len(inputs), batch_size):
            end = start + batch_size
            batch = torch.as_tensor(inputs[start:end], dtype=torch.float32, device=device)
            losses = F.cross_entropy(model(batch), classes[start:end], reduction='none')
            total += (losses * weights[start:end]).sum().item()
    return total / weights.sum().item()


def build_importance_table(final_runs, split, columns):
    groups = build_feature_groups(columns)
    rows = []
    for model_name, runs in final_runs.items():
        for trained in runs:
            inputs = split.windows() if trained.spec.sequence else split.features
            batch_size = trained.hyperparameters.batch_size
            base_loss = cross_entropy_on_inputs(trained.model, inputs, split, batch_size)
            rng = np.random.default_rng(trained.seed)
            for group_name, indices in groups.items():
                shuffled = inputs.copy()
                shuffled[..., indices] = inputs[rng.permutation(len(inputs))][..., indices]
                loss = cross_entropy_on_inputs(trained.model, shuffled, split, batch_size)
                rows.append({'model': model_name, 'feature_group': group_name, 'loss_increase': loss - base_loss})
    table = pd.DataFrame(rows).groupby(['model', 'feature_group'], sort=False)['loss_increase'].agg(['mean', 'std'])
    return table.reset_index().rename(columns={'mean': 'loss_increase_mean', 'std': 'loss_increase_std'})


exporter.save(
    'feature_importance', build_importance_table(main_result.final_runs, main_result.fold_datas[-1].validation, selected_columns),
    'Bars of loss_increase_mean by feature group for one model', 'Permutation importance on validation (main setup)',
)

In [ ]:
CONFIDENCE_BINS = np.round(np.arange(0.30, 1.0001, 0.05), 2)
SPREAD_GROUPS = 5
MIN_BAND_EVENTS = 30  # bands with fewer events are saved but not plotted


def pool(splits, attribute):
    return np.concatenate([getattr(split, attribute) for split in splits])


def labelled(table, **labels):
    """The table with label columns added at the front."""
    return table.assign(**labels)[[*labels, *table.columns]]


def evaluation_groups(result):
    """Validation, plus the pooled test files once unlocked, as (splits, calibrated probabilities by model)."""
    groups = {'validation': ([result.fold_datas[-1].validation], result.validation_probabilities)}
    if result.name in TEST_RESULTS:
        test = TEST_RESULTS[result.name]
        groups['test'] = (
            list(test['splits'].values()),
            {model_name: np.vstack([test['probabilities'][name][model_name] for name in test['splits']]) for model_name in result.final_runs},
        )
    return groups


def build_reliability_table(probabilities, splits):
    confidence = probabilities.max(axis=1)
    frame = pd.DataFrame({
        'band': pd.cut(confidence, CONFIDENCE_BINS, include_lowest=True),
        'confidence': confidence,
        'correct': probabilities.argmax(axis=1) == pool(splits, 'class_indices'),
    })
    table = frame.groupby('band', observed=True).agg(
        mean_confidence=('confidence', 'mean'), accuracy=('correct', 'mean'), events=('correct', 'size'),
    ).reset_index()
    table['band'] = table['band'].astype(str)
    return table


def build_spread_table(probabilities, splits):
    """Trades only, in equal-sized spread groups from 1 (tightest) up."""
    predicted = probabilities.argmax(axis=1)
    traded = predicted != 1
    if not traded.any():
        return pd.DataFrame()
    costs = pool(splits, 'costs')[traded]
    frame = pd.DataFrame({
        'spread_group': pd.qcut(costs, SPREAD_GROUPS, labels=False, duplicates='drop') + 1,
        'spread_bps': costs * 1e4,
        'correct': predicted[traded] == pool(splits, 'class_indices')[traded],
        'gross_bps': (predicted[traded] - 1) * pool(splits, 'gross_returns')[traded] * 1e4,
    })
    table = frame.groupby('spread_group').agg(
        trades=('correct', 'size'), mean_spread_bps=('spread_bps', 'mean'),
        direction_precision=('correct', 'mean'), gross_bps=('gross_bps', 'mean'),
    ).reset_index()
    table['net_bps'] = table['gross_bps'] - table['mean_spread_bps']
    return table


def build_example_paths(split, setup_config):
    """The mid-price path after the first validation event of each class, with its barriers."""
    widths = barrier_widths(train_volatility, train_file.relative_spread, selected_barrier, setup_config)
    rows = []
    for label, class_name in zip((-1, 0, 1), CLASS_NAMES):
        index = int(np.flatnonzero(split.labels == label)[0])
        t0, t1 = int(split.event_ticks[index]), int(split.event_ends[index])
        for offset in range(setup_config.holding_period + 1):
            rows.append({
                'label': class_name, 'event_tick': t0, 'tick_offset': offset,
                'return_bps': np.log(train_file.mid_price[t0 + offset] / train_file.mid_price[t0]) * 1e4,
                'upper_barrier_bps': widths[t0] * 1e4, 'lower_barrier_bps': -widths[t0] * 1e4,
                'label_decided_here': offset == t1 - t0,
            })
    return pd.DataFrame(rows)


def plot_all_cumulative_returns(kind):
    """One panel per setup: every model's ensemble and the always-up and always-down baselines, on validation."""
    fig, axes = plt.subplots(1, len(SETUPS), figsize=(5 * len(SETUPS), 4.2), squeeze=False)
    for ax, (setup_name, result) in zip(axes[0], SETUPS.items()):
        table = build_cumulative_returns(result.validation_predictions, 'validation', result.fold_datas[-1].validation)
        for color, spec in zip(SERIES_COLORS, MODEL_SPECS):
            ax.plot(table['event'], table[f'{spec.name}_ensemble_{kind}_bps'], color=color, label=spec.name)
        for style, baseline in (('--', 'always_up'), (':', 'always_down')):
            ax.plot(table['event'], table[f'{baseline}_{kind}_bps'], color=REFERENCE_COLOR, linestyle=style, label=baseline)
        ax.axhline(0, color=TEXT_COLOR, linewidth=0.8)
        ax.set_title(setup_name)
        ax.set_xlabel('Validation event, in time order within each stock')
    axes[0][0].set_ylabel(f'Cumulative {kind} return (bps)')
    axes[0][0].legend(frameon=False, fontsize=8)
    fig.suptitle(f'Every model, {kind} cumulative return on validation', x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()


def plot_reliability(table):
    setup_names = list(dict.fromkeys(table['setup']))
    fig, axes = plt.subplots(1, len(setup_names), figsize=(4.2 * len(setup_names), 4), sharey=True, squeeze=False)
    for ax, setup_name in zip(axes[0], setup_names):
        rows = table[(table['setup'] == setup_name) & (table['events'] >= MIN_BAND_EVENTS)]
        for color, model_name in zip(SERIES_COLORS, dict.fromkeys(rows['model'])):
            line = rows[rows['model'] == model_name]
            ax.plot(line['mean_confidence'], line['accuracy'], marker='o', color=color, label=model_name)
        ax.plot([0.3, 1], [0.3, 1], color=REFERENCE_COLOR, linestyle='--', linewidth=1, label='Perfectly calibrated')
        ax.set_title(setup_name)
        ax.set_xlabel('Confidence')
    axes[0][0].set_ylabel('Share correct')
    axes[0][0].legend(frameon=False, fontsize=8)
    fig.suptitle('Calibration on validation', x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()


def plot_example_paths(table):
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(13, 3.8), sharey=True)
    for ax, class_name in zip(axes, CLASS_NAMES):
        path = table[table['label'] == class_name]
        decided = path[path['label_decided_here']]
        ax.plot(path['tick_offset'], path['return_bps'], color=TRAIN_COLOR)
        for column in ('upper_barrier_bps', 'lower_barrier_bps'):
            ax.axhline(path[column].iloc[0], color=VALIDATION_COLOR, linestyle='--', linewidth=1.5)
        ax.axvline(decided['tick_offset'].iloc[0], color=REFERENCE_COLOR, linestyle=':', linewidth=1.5)
        ax.set_title(f'Labeled {class_name}')
        ax.set_xlabel('Ticks after the event')
    axes[0].set_ylabel('Mid-price change (bps)')
    fig.suptitle('Triple barrier examples: dashed lines are the barriers, dotted is where the label was decided', x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    plt.show()


plot_all_cumulative_returns('gross')
plot_all_cumulative_returns('net')

reliability_tables = []
spread_tables = []
for setup_name, result in SETUPS.items():
    for group, (splits, probabilities) in evaluation_groups(result).items():
        for model_name in dict.fromkeys([result.selected_model, 'logistic_regression']):
            labels = {'setup': setup_name, 'model': model_name, 'group': group}
            reliability_tables.append(labelled(build_reliability_table(probabilities[model_name], splits), **labels))
            spread_tables.append(labelled(build_spread_table(probabilities[model_name], splits), **labels))
reliability = pd.concat(reliability_tables, ignore_index=True)
exporter.save('calibration_reliability', reliability, 'Scatter of accuracy against mean_confidence, with a diagonal line', show=False)
plot_reliability(reliability[reliability['group'] == 'validation'])
exporter.save(
    'results_by_spread', pd.concat(spread_tables, ignore_index=True),
    'Clustered bars of gross_bps and net_bps by spread_group', 'Trading results by spread group (1 = tightest)',
)

example_paths = build_example_paths(main_result.fold_datas[-1].validation, config)
exporter.save('example_label_paths', example_paths, 'Line chart of return_bps by tick_offset, one per label, with the barriers', show=False)
plot_example_paths(example_paths)

## 10. Key results and export

Headline table, `run_info.json` (settings, selections, versions) and a zip of all CSVs.

In [ ]:
key_rows = []
for setup_name, result in SETUPS.items():
    groups = {'validation': VALIDATION_SCORES[setup_name]}
    if setup_name in TEST_RESULTS:
        groups['test'] = TEST_RESULTS[setup_name]['scores']
    for group, scores in groups.items():
        summary = summarize_scores(scores)
        for model_name in dict.fromkeys(['majority_class', 'logistic_regression', result.selected_model]):
            row_name = model_name if model_name in BASELINES else f'{model_name}_ensemble'
            key_rows.append({
                'setup': setup_name, 'group': group, 'model': model_name,
                'selected': model_name == result.selected_model,
                **{metric: summary.loc[row_name, f'{metric}_mean'] for metric in SCORE_COLUMNS},
            })
exporter.save('key_results', pd.DataFrame(key_rows), 'Table for the slides', 'Key results')

run_info = {
    'created': datetime.now().isoformat(),
    'data_zip_sha256': hashlib.sha256(Path('data.zip').read_bytes()).hexdigest() if Path('data.zip').exists() else None,
    'pipeline_config': asdict(config),
    'cost_aware_config': asdict(cost_config),
    'model_config': asdict(model_config),
    'selected_d': selected_d,
    'selected_barrier_multiplier': selected_barrier,
    'selected_scaling': selected_scaling,
    'selected_feature_set': selected_feature_set,
    'setups': {
        name: {
            'selected_model': result.selected_model, 'highest_scoring_model': result.highest_scoring_model,
            'temperatures': result.temperatures,
            'tuned_settings': {model: values['best_params'] for model, values in result.tuning_results.items()},
        }
        for name, result in SETUPS.items()
    },
    'test_evaluated': EVALUATE_TEST,
    'versions': {
        'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
        'torch': torch.__version__, 'sklearn': sklearn.__version__, 'statsmodels': statsmodels.__version__,
        'optuna': optuna.__version__,
    },
}
(context.run_dir / 'run_info.json').write_text(json.dumps(run_info, indent=2, default=str))
zip_path = exporter.finish()
if IN_COLAB and DOWNLOAD_EXPORTS:
    files.download(zip_path)